# ID Minter Invoke And Verify Harness

This notebook samples root work source identifiers from `works-source-2025-10-02`, invokes the new id_minter Lambda for a smoke test and larger verification batches, reads back results from `works-identified-2026-03-06`, and compares them with the current `works-identified-2025-10-02` index.

The comparison is intentionally narrow: it compares `data` between the new and baseline identified documents, while checking root `state.canonicalId` separately.

The comparison currently treats two classes of differences as expected:

- canonical ID drift, because the v2 identifiers database behind the new Python id_minter was migrated from a daily snapshot and may legitimately mint a different root canonical ID from the current baseline index
- explicit `null` fields present in the new Python-produced document when the baseline document omits the same field, because the older Scala-produced path often dropped null-valued fields during serialisation

Other differences are treated as real mismatches. 

The Scala decode checks reuse the target documents already fetched during comparison, so they do not need to download the same identified docs again.

Run the cells in order and start with the smoke test before moving to the 1k verification batch.

In [ ]:
import json
import os
import random
import subprocess
import sys
import uuid
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter
from typing import Any, NoReturn

import boto3
from botocore.exceptions import UnauthorizedSSOTokenError

SRC_ROOT = Path.cwd().resolve().parent / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

try:
    import pandas as pd
except ImportError:
    pd = None

from utils.aws import dicts_from_s3_jsonl
from utils.elasticsearch import get_client, get_standard_index_name
from utils.models.manifests import StepManifest

os.environ.setdefault("AWS_PROFILE", "platform-developer")

AWS_PROFILE = os.environ["AWS_PROFILE"]
REGION = "eu-west-1"
PIPELINE_DATE = "2025-10-02"
TARGET_INDEX_DATE_SUFFIX = "2026-03-06"
LAMBDA_ARN = "arn:aws:lambda:eu-west-1:760097843905:function:catalogue-2025-10-02-id-minter"
SOURCE_INDEX = get_standard_index_name("works-source", PIPELINE_DATE)
BASELINE_IDENTIFIED_INDEX = get_standard_index_name("works-identified", PIPELINE_DATE)
TARGET_IDENTIFIED_INDEX = get_standard_index_name("works-identified", TARGET_INDEX_DATE_SUFFIX)
VERIFICATION_BATCH_SIZE = 1000
DEFAULT_SAMPLE_BUFFER = 250
REPO_ROOT = Path.cwd().resolve().parents[1]
SCALA_DECODE_MAIN_CLASS = "weco.pipeline.merger.tools.IdentifiedIndexDecodeChecker"
SCALA_DECODE_DIR = REPO_ROOT / ".scratch" / "id_minter_decode_checks"
SCALA_DECODE_DIR.mkdir(parents=True, exist_ok=True)

session = boto3.Session(profile_name=AWS_PROFILE, region_name=REGION)
lambda_client = session.client("lambda", region_name=REGION)
_source_es_client = None
_baseline_es_client = None
_target_es_client = None


def raise_sso_login_error(error: UnauthorizedSSOTokenError) -> NoReturn:
    raise RuntimeError(
        f"AWS SSO session for profile {AWS_PROFILE} has expired. "
        f"Run `aws sso login --profile {AWS_PROFILE}` and re-run the cell."
    ) from error


def get_source_es_client() -> Any:
    global _source_es_client
    if _source_es_client is None:
        try:
            _source_es_client = get_client("id_minter", PIPELINE_DATE, "public")
        except UnauthorizedSSOTokenError as error:
            raise_sso_login_error(error)
    return _source_es_client


def get_baseline_es_client() -> Any:
    global _baseline_es_client
    if _baseline_es_client is None:
        try:
            _baseline_es_client = get_client("id_minter", PIPELINE_DATE, "public")
        except UnauthorizedSSOTokenError as error:
            raise_sso_login_error(error)
    return _baseline_es_client


def get_target_es_client() -> Any:
    global _target_es_client
    if _target_es_client is None:
        try:
            _target_es_client = get_client("id_minter", PIPELINE_DATE, "public")
        except UnauthorizedSSOTokenError as error:
            raise_sso_login_error(error)
    return _target_es_client


RUN_PREFIX = f"id-minter-verify-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"

print(f"AWS profile:             {AWS_PROFILE}")
print(f"Lambda ARN:              {LAMBDA_ARN}")
print(f"Source index:            {SOURCE_INDEX}")
print(f"Baseline identified:     {BASELINE_IDENTIFIED_INDEX}")
print(f"Target identified:       {TARGET_IDENTIFIED_INDEX}")
print(f"Verification batch size: {VERIFICATION_BATCH_SIZE}")
print(f"Run prefix:              {RUN_PREFIX}")
print(f"Repo root:               {REPO_ROOT}")
print("Elasticsearch clients are created lazily when sampling or comparing.")

In [ ]:
ROOT_SOURCE_FIELDS = [
    "state.sourceIdentifier",
    "state.canonicalId",
    "data",
]

_baseline_doc_cache: dict[str, dict[str, Any] | None] = {}


def build_source_identifier_string(source_identifier: dict[str, Any]) -> str:
    return (
        f"{source_identifier['ontologyType']}"
        f"[{source_identifier['identifierType']['id']}/{source_identifier['value']}]"
    )


def build_job_id(label: str | None = None) -> str:
    suffix = uuid.uuid4().hex[:8]
    if label:
        return f"{RUN_PREFIX}-{label}-{suffix}"
    return f"{RUN_PREFIX}-{suffix}"


def manifest_location_to_s3_uri(location: Any) -> str:
    return f"s3://{location.bucket}/{location.key}"


def read_manifest_batches(manifest: StepManifest | None) -> dict[str, list[dict[str, Any]]]:
    if manifest is None:
        return {"successes": [], "failures": []}

    try:
        batches = {
            "successes": dicts_from_s3_jsonl(
                manifest_location_to_s3_uri(manifest.successes.batch_file_location)
            )
        }
        if manifest.failures is None:
            batches["failures"] = []
        else:
            batches["failures"] = dicts_from_s3_jsonl(
                manifest_location_to_s3_uri(manifest.failures.error_file_location)
            )
        return batches
    except UnauthorizedSSOTokenError as error:
        raise_sso_login_error(error)


def build_root_source_identifier_query(source_identifier: dict[str, Any]) -> dict[str, Any]:
    return {
        "bool": {
            "filter": [
                {
                    "term": {
                        "state.sourceIdentifier.ontologyType": source_identifier[
                            "ontologyType"
                        ]
                    }
                },
                {
                    "term": {
                        "state.sourceIdentifier.identifierType.id": source_identifier[
                            "identifierType"
                        ]["id"]
                    }
                },
                {
                    "term": {
                        "state.sourceIdentifier.value": source_identifier["value"]
                    }
                },
            ]
        }
    }


def extract_root_source_identifier(doc_source: dict[str, Any]) -> dict[str, Any] | None:
    return doc_source.get("state", {}).get("sourceIdentifier")


def flatten_manifest_canonical_ids(
    manifest_batches: dict[str, list[dict[str, Any]]],
) -> list[str]:
    canonical_ids: list[str] = []
    for line in manifest_batches["successes"]:
        canonical_ids.extend(line.get("canonicalIds", []))
    return canonical_ids


def fetch_target_docs_by_canonical_id(
    canonical_ids: list[str],
    source_fields: list[str] | None = None,
) -> dict[str, Any]:
    unique_ids = list(dict.fromkeys(canonical_ids))
    started = perf_counter()
    docs_by_canonical_id: dict[str, dict[str, Any]] = {}

    if not unique_ids:
        return {
            "docs_by_canonical_id": docs_by_canonical_id,
            "docs_by_source_identifier": {},
            "query_count": 0,
            "elapsed_seconds": perf_counter() - started,
            "missing_canonical_ids": [],
        }

    try:
        kwargs: dict[str, Any] = {
            "index": TARGET_IDENTIFIED_INDEX,
            "body": {"ids": unique_ids},
        }
        if source_fields is not None:
            kwargs["_source"] = source_fields
        response = get_target_es_client().mget(**kwargs)
    except UnauthorizedSSOTokenError as error:
        raise_sso_login_error(error)

    docs_by_canonical_id = {
        doc["_id"]: doc
        for doc in response["docs"]
        if doc.get("found") and doc.get("_source")
    }
    missing_ids = [cid for cid in unique_ids if cid not in docs_by_canonical_id]

    docs_by_source_identifier: dict[str, dict[str, Any]] = {}
    for doc in docs_by_canonical_id.values():
        source_identifier = extract_root_source_identifier(doc["_source"])
        if source_identifier is None:
            continue
        docs_by_source_identifier[build_source_identifier_string(source_identifier)] = doc

    return {
        "docs_by_canonical_id": docs_by_canonical_id,
        "docs_by_source_identifier": docs_by_source_identifier,
        "query_count": 1,
        "elapsed_seconds": perf_counter() - started,
        "missing_canonical_ids": missing_ids,
    }


def fetch_baseline_docs(
    source_identifiers: list[dict[str, Any]],
) -> dict[str, Any]:
    requested_keys = [build_source_identifier_string(source_identifier) for source_identifier in source_identifiers]
    missing_source_identifiers = [
        source_identifier
        for source_identifier in source_identifiers
        if build_source_identifier_string(source_identifier) not in _baseline_doc_cache
    ]

    started = perf_counter()
    query_count = 0
    if missing_source_identifiers:
        query_count = 1
        try:
            response = get_baseline_es_client().search(
                index=BASELINE_IDENTIFIED_INDEX,
                body={
                    "size": len(missing_source_identifiers),
                    "_source": ROOT_SOURCE_FIELDS,
                    "query": {
                        "bool": {
                            "should": [
                                build_root_source_identifier_query(source_identifier)
                                for source_identifier in missing_source_identifiers
                            ],
                            "minimum_should_match": 1,
                        }
                    },
                },
            )
        except UnauthorizedSSOTokenError as error:
            raise_sso_login_error(error)

        fetched_docs: dict[str, dict[str, Any]] = {}
        for hit in response["hits"]["hits"]:
            root_source_identifier = extract_root_source_identifier(hit["_source"])
            if root_source_identifier is None:
                continue
            fetched_docs[build_source_identifier_string(root_source_identifier)] = hit

        for source_identifier in missing_source_identifiers:
            key = build_source_identifier_string(source_identifier)
            _baseline_doc_cache[key] = fetched_docs.get(key)

    docs_by_source_identifier = {
        key: _baseline_doc_cache.get(key) for key in requested_keys
    }
    cache_misses = sum(
        1 for key in requested_keys if docs_by_source_identifier.get(key) is None
    )

    return {
        "docs_by_source_identifier": docs_by_source_identifier,
        "query_count": query_count,
        "elapsed_seconds": perf_counter() - started,
        "cache_hits": len(requested_keys) - len(missing_source_identifiers),
        "cache_misses": cache_misses,
        "cache_writes": len(missing_source_identifiers),
    }


def sample_source_work_candidates(
    desired_count: int,
    oversample_buffer: int = DEFAULT_SAMPLE_BUFFER,
    exclude_today: bool = True,
    seed: int | None = None,
) -> list[dict[str, Any]]:
    today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    seed = random.randint(0, 2**31) if seed is None else seed
    try:
        response = get_source_es_client().search(
            index=SOURCE_INDEX,
            body={
                "_source": ["state.sourceIdentifier", "state.sourceModifiedTime"],
                "query": {
                    "function_score": {
                        "query": {"match_all": {}},
                        "random_score": {"seed": seed, "field": "_seq_no"},
                    }
                },
                "size": desired_count + oversample_buffer,
            },
        )
    except UnauthorizedSSOTokenError as error:
        raise_sso_login_error(error)

    candidates: list[dict[str, Any]] = []
    skipped = Counter()
    seen_source_identifiers: set[str] = set()

    for hit in response["hits"]["hits"]:
        source_doc = hit["_source"]
        state = source_doc.get("state", {})
        source_identifier = state.get("sourceIdentifier")
        if not source_identifier:
            skipped["missing_root_source_identifier"] += 1
            continue

        source_identifier_str = build_source_identifier_string(source_identifier)
        if source_identifier_str in seen_source_identifiers:
            skipped["duplicate_root_source_identifier"] += 1
            continue

        source_modified_time = state.get("sourceModifiedTime", "")
        if exclude_today and source_modified_time.startswith(today):
            skipped["modified_today"] += 1
            continue

        seen_source_identifiers.add(source_identifier_str)
        candidates.append(
            {
                "source_doc_id": hit["_id"],
                "source_identifier": source_identifier,
                "source_identifier_str": source_identifier_str,
                "source_modified_time": source_modified_time,
            }
        )
        if len(candidates) >= desired_count:
            break

    print(
        f"Sampled {len(candidates)} candidates from {SOURCE_INDEX} "
        f"(seed={seed}, skipped={dict(skipped)})"
    )
    return candidates


def invoke_id_minter(
    source_identifiers: list[str],
    label: str | None = None,
) -> dict[str, Any]:
    job_id = build_job_id(label)
    payload = {
        "source_identifiers": source_identifiers,
        "job_id": job_id,
    }

    started = perf_counter()
    try:
        response = lambda_client.invoke(
            FunctionName=LAMBDA_ARN,
            InvocationType="RequestResponse",
            Payload=json.dumps(payload).encode("utf-8"),
        )
    except UnauthorizedSSOTokenError as error:
        raise_sso_login_error(error)
    elapsed_seconds = perf_counter() - started
    raw_payload = response["Payload"].read().decode("utf-8")

    parsed_payload: dict[str, Any] | None
    if raw_payload.strip():
        parsed_payload = json.loads(raw_payload)
    else:
        parsed_payload = None

    manifest = None
    if response.get("FunctionError") is None and parsed_payload is not None:
        manifest = StepManifest.model_validate(parsed_payload)

    return {
        "job_id": job_id,
        "request_payload": payload,
        "status_code": response["StatusCode"],
        "function_error": response.get("FunctionError"),
        "elapsed_seconds": elapsed_seconds,
        "raw_payload": raw_payload,
        "parsed_payload": parsed_payload,
        "manifest": manifest,
    }


def deep_diff(
    ours: object,
    theirs: object,
    path: str = "",
) -> list[tuple[str, str, object, object]]:
    diffs: list[tuple[str, str, object, object]] = []
    if isinstance(ours, dict) and isinstance(theirs, dict):
        for key in sorted(set(ours) | set(theirs)):
            child_path = f"{path}.{key}" if path else key
            if key not in theirs:
                diffs.append((child_path, "UNEXPECTED:key_only_in_ours", ours[key], None))
            elif key not in ours:
                diffs.append(
                    (child_path, "UNEXPECTED:key_only_in_theirs", None, theirs[key])
                )
            else:
                diffs.extend(deep_diff(ours[key], theirs[key], child_path))
        return diffs

    if isinstance(ours, list) and isinstance(theirs, list):
        if len(ours) != len(theirs):
            diffs.append((path, "UNEXPECTED:list_length", len(ours), len(theirs)))
        for index, (left_value, right_value) in enumerate(zip(ours, theirs)):
            diffs.extend(deep_diff(left_value, right_value, f"{path}[{index}]"))
        return diffs

    if ours != theirs:
        diffs.append((path, "UNEXPECTED:value_mismatch", ours, theirs))
    return diffs


def classify_diff(
    path: str,
    category: str,
    ours: object,
) -> str:
    if category == "UNEXPECTED:key_only_in_ours" and ours is None:
        return "expected:scala_null_dropped"
    return category


def compare_identified_sources(
    candidate: dict[str, Any],
    target_hit: dict[str, Any] | None,
    baseline_hit: dict[str, Any] | None,
) -> dict[str, Any]:
    category_counts = Counter()
    unexpected_diffs: list[dict[str, Any]] = []

    if target_hit is None or baseline_hit is None:
        if target_hit is None:
            category_counts["missing:target"] += 1
        if baseline_hit is None:
            category_counts["missing:baseline"] += 1
        return {
            "source_identifier_str": candidate["source_identifier_str"],
            "target_hit": target_hit,
            "baseline_hit": baseline_hit,
            "category_counts": dict(category_counts),
            "unexpected_diffs": unexpected_diffs,
        }

    target_source = target_hit["_source"]
    baseline_source = baseline_hit["_source"]

    target_canonical_id = target_source.get("state", {}).get("canonicalId")
    baseline_canonical_id = baseline_source.get("state", {}).get("canonicalId")
    if target_canonical_id != baseline_canonical_id:
        category_counts["expected:canonical_id_drift"] += 1

    target_data = target_source.get("data", {})
    baseline_data = baseline_source.get("data", {})
    for path, raw_category, ours, theirs in deep_diff(target_data, baseline_data, path="data"):
        category = classify_diff(path, raw_category, ours)
        category_counts[category] += 1
        if category.startswith("UNEXPECTED"):
            unexpected_diffs.append(
                {
                    "source_identifier_str": candidate["source_identifier_str"],
                    "path": path,
                    "category": category,
                    "ours": ours,
                    "theirs": theirs,
                }
            )

    return {
        "source_identifier_str": candidate["source_identifier_str"],
        "target_hit": target_hit,
        "baseline_hit": baseline_hit,
        "category_counts": dict(category_counts),
        "unexpected_diffs": unexpected_diffs,
    }


def summarize_comparisons(comparisons: list[dict[str, Any]]) -> dict[str, int]:
    totals = Counter()
    for comparison in comparisons:
        for category, count in comparison["category_counts"].items():
            totals[category] += count
        if comparison["unexpected_diffs"]:
            totals["unexpected_docs"] += 1
    return dict(totals)


def run_single_batch(
    candidates: list[dict[str, Any]],
    label: str | None = None,
) -> dict[str, Any]:
    timings_started = perf_counter()
    source_identifiers = [candidate["source_identifier_str"] for candidate in candidates]

    invocation = invoke_id_minter(source_identifiers, label=label)

    manifest_started = perf_counter()
    manifest_batches = read_manifest_batches(invocation["manifest"])
    manifest_elapsed_seconds = perf_counter() - manifest_started

    target_canonical_ids = flatten_manifest_canonical_ids(manifest_batches)
    target_fetch = fetch_target_docs_by_canonical_id(target_canonical_ids)

    baseline_fetch = fetch_baseline_docs(
        [candidate["source_identifier"] for candidate in candidates]
    )

    diff_started = perf_counter()
    comparisons = [
        compare_identified_sources(
            candidate,
            target_fetch["docs_by_source_identifier"].get(
                candidate["source_identifier_str"]
            ),
            baseline_fetch["docs_by_source_identifier"].get(
                candidate["source_identifier_str"]
            ),
        )
        for candidate in candidates
    ]
    diff_elapsed_seconds = perf_counter() - diff_started

    comparison_summary = summarize_comparisons(comparisons)
    batch_summary = {
        "job_id": invocation["job_id"],
        "batch_size": len(candidates),
        "status_code": invocation["status_code"],
        "function_error": invocation["function_error"],
        "lambda_seconds": round(invocation["elapsed_seconds"], 3),
        "manifest_seconds": round(manifest_elapsed_seconds, 3),
        "target_fetch_seconds": round(target_fetch["elapsed_seconds"], 3),
        "baseline_fetch_seconds": round(baseline_fetch["elapsed_seconds"], 3),
        "diff_seconds": round(diff_elapsed_seconds, 3),
        "total_post_lambda_seconds": round(
            perf_counter() - timings_started - invocation["elapsed_seconds"],
            3,
        ),
        "manifest_success_count": invocation["manifest"].successes.count
        if invocation["manifest"] is not None
        else 0,
        "manifest_failure_count": invocation["manifest"].failures.count
        if invocation["manifest"] is not None and invocation["manifest"].failures is not None
        else 0,
        "success_batch_line_count": len(manifest_batches["successes"]),
        "failure_batch_line_count": len(manifest_batches["failures"]),
        "target_manifest_canonical_ids": len(target_canonical_ids),
        "target_docs_found": len(target_fetch["docs_by_canonical_id"]),
        "target_missing_docs": len(target_fetch["missing_canonical_ids"]),
        "target_query_count": target_fetch["query_count"],
        "baseline_query_count": baseline_fetch["query_count"],
        "baseline_cache_hits": baseline_fetch["cache_hits"],
        "baseline_cache_writes": baseline_fetch["cache_writes"],
        "baseline_docs_found": sum(
            1
            for candidate in candidates
            if baseline_fetch["docs_by_source_identifier"].get(
                candidate["source_identifier_str"]
            )
            is not None
        ),
    }
    batch_summary.update(comparison_summary)

    return {
        "summary": batch_summary,
        "invocation": invocation,
        "manifest_batches": manifest_batches,
        "target_fetch": target_fetch,
        "baseline_fetch": baseline_fetch,
        "comparisons": comparisons,
        "unexpected_diffs": [
            diff
            for comparison in comparisons
            for diff in comparison["unexpected_diffs"]
        ],
    }


def run_verification_batches(
    candidate_pool: list[dict[str, Any]],
    batch_size: int = VERIFICATION_BATCH_SIZE,
    max_batches: int | None = None,
) -> dict[str, Any]:
    if batch_size <= 0:
        raise ValueError("batch_size must be positive")

    available_batches = len(candidate_pool) // batch_size
    if available_batches == 0:
        raise ValueError(
            f"Need at least {batch_size} candidates, only have {len(candidate_pool)}"
        )

    batch_count = available_batches if max_batches is None else min(max_batches, available_batches)
    runs = []
    for batch_index in range(batch_count):
        start = batch_index * batch_size
        batch_candidates = candidate_pool[start : start + batch_size]
        runs.append(
            run_single_batch(
                batch_candidates,
                label=f"verify-{batch_size}-{batch_index + 1:02d}",
            )
        )

    metrics = [run["summary"] for run in runs]
    return {"runs": runs, "metrics": metrics}


def results_frame(metrics: list[dict[str, Any]]) -> Any:
    if pd is None:
        return metrics
    return pd.DataFrame(metrics).fillna(0)

In [ ]:
MATCHER_SOURCE_FIELDS = ["state", "version", "type"]


def get_batch_runs(result: dict[str, Any]) -> list[dict[str, Any]]:
    if "runs" in result:
        return result["runs"]
    return [result]


def collect_manifest_canonical_ids(result: dict[str, Any]) -> list[str]:
    canonical_ids: list[str] = []
    for batch_run in get_batch_runs(result):
        canonical_ids.extend(flatten_manifest_canonical_ids(batch_run["manifest_batches"]))
    return list(dict.fromkeys(canonical_ids))


def collect_target_docs_by_canonical_id(
    result: dict[str, Any],
) -> dict[str, dict[str, Any]]:
    canonical_ids = collect_manifest_canonical_ids(result)
    docs_by_canonical_id: dict[str, dict[str, Any]] = {}
    for batch_run in get_batch_runs(result):
        batch_docs = batch_run["target_fetch"]["docs_by_canonical_id"]
        for canonical_id in canonical_ids:
            doc = batch_docs.get(canonical_id)
            if doc is not None and canonical_id not in docs_by_canonical_id:
                docs_by_canonical_id[canonical_id] = doc
    return docs_by_canonical_id


def select_source_fields(
    document: dict[str, Any],
    source_fields: list[str] | None,
) -> dict[str, Any]:
    if source_fields is None:
        return document
    return {field: document[field] for field in source_fields if field in document}


def build_scala_decode_input_records(
    docs_by_canonical_id: dict[str, dict[str, Any]],
    source_fields: list[str] | None = None,
) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    for canonical_id, doc in docs_by_canonical_id.items():
        source_identifier = extract_root_source_identifier(doc["_source"])
        records.append(
            {
                "docId": canonical_id,
                "sourceIdentifier": (
                    build_source_identifier_string(source_identifier)
                    if source_identifier is not None
                    else None
                ),
                "document": select_source_fields(doc["_source"], source_fields),
            }
        )
    return records


def write_scala_decode_input(
    records: list[dict[str, Any]],
    mode: str,
    label: str | None = None,
) -> Path:
    suffix = uuid.uuid4().hex[:8]
    label_prefix = f"{label}-" if label else ""
    input_path = SCALA_DECODE_DIR / f"{label_prefix}{mode}-{suffix}.jsonl"
    lines = [json.dumps(record) for record in records]
    input_path.write_text("\n".join(lines) + ("\n" if lines else ""))
    return input_path


def run_scala_decode_checker(
    mode: str,
    input_path: Path,
    output_path: Path | None = None,
) -> dict[str, Any]:
    output_path = output_path or SCALA_DECODE_DIR / f"{input_path.stem}-summary.json"
    command = [
        "sbt",
        "project merger",
        (
            f"runMain {SCALA_DECODE_MAIN_CLASS} "
            f"--mode {mode} --input {input_path} --output {output_path}"
        ),
    ]
    completed = subprocess.run(
        command,
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(
            "Scala decode checker failed.\n"
            f"Command: {' '.join(command)}\n"
            f"Exit code: {completed.returncode}\n"
            f"STDOUT:\n{completed.stdout[-4000:]}\n"
            f"STDERR:\n{completed.stderr[-4000:]}"
        )

    summary = json.loads(output_path.read_text())
    return {
        "mode": mode,
        "input_path": input_path,
        "output_path": output_path,
        "summary": summary,
        "stdout": completed.stdout,
        "stderr": completed.stderr,
    }


def run_scala_decode_from_result(
    result: dict[str, Any],
    mode: str,
    label: str | None = None,
) -> dict[str, Any]:
    canonical_ids = collect_manifest_canonical_ids(result)
    docs_by_canonical_id = collect_target_docs_by_canonical_id(result)
    source_fields = MATCHER_SOURCE_FIELDS if mode == "matcher" else None
    missing_canonical_ids = [
        canonical_id for canonical_id in canonical_ids if canonical_id not in docs_by_canonical_id
    ]
    records = build_scala_decode_input_records(
        docs_by_canonical_id,
        source_fields=source_fields,
    )
    input_path = write_scala_decode_input(records, mode=mode, label=label)
    decode_result = run_scala_decode_checker(mode=mode, input_path=input_path)
    decode_result["record_count"] = len(records)
    decode_result["missing_canonical_ids"] = missing_canonical_ids
    decode_result["reused_target_docs"] = True
    return decode_result


def decode_failures_frame(decode_result: dict[str, Any]) -> Any:
    failures = decode_result["summary"].get("failures", [])
    if pd is None:
        return failures
    return pd.DataFrame(failures)


In [ ]:
# Re-run this cell after changing the Python id_minter so `verify_1000`
# reflects freshly written identified docs rather than an older batch result.

candidate_pool = sample_source_work_candidates(
    desired_count=1 + VERIFICATION_BATCH_SIZE,
    oversample_buffer=max(DEFAULT_SAMPLE_BUFFER, VERIFICATION_BATCH_SIZE // 4),
)

# candidate_pool[:3]

# Smoke test:
# smoke = run_single_batch(
#     candidate_pool[:1],
#     label="smoke",
# )
# smoke["summary"]
# smoke["unexpected_diffs"][:10]

# Verification batch of 1k:
verify_1000 = run_single_batch(
    candidate_pool[1 : 1 + VERIFICATION_BATCH_SIZE],
    label="verify-1000",
)
verify_1000["summary"]
verify_1000["unexpected_diffs"][:10]

In [ ]:
# Scala matcher decode check against docs written to the new identified index.
# This reuses docs already downloaded during comparison in `run_single_batch`.
# Use a freshly rerun batch result, for example `smoke` or `verify_1000`,
# after re-running the batch cell above.

decode_source = verify_1000
matcher_decode = run_scala_decode_from_result(
    decode_source,
    mode="matcher",
    label="matcher-decode",
)
matcher_decode["summary"]
decode_failures_frame(matcher_decode)


In [ ]:
# Scala merger decode check against full docs from the new identified index.
# This is a stricter compatibility check than matcher mode because it decodes full Work[Identified].
# It reuses the full target docs already fetched during comparison.
# Use a freshly rerun batch result, for example `smoke` or `verify_1000`,
# after re-running the batch cell above.

decode_source = verify_1000
merger_decode = run_scala_decode_from_result(
    decode_source,
    mode="merger",
    label="merger-decode",
)
merger_decode["summary"]
decode_failures_frame(merger_decode)
